## 준비

TensorFlow는 임포트할 때 CPU 최적화/GPU 관련 경고가 나올 수 있음.

In [3]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3" #정보성 로그 끄겠다.
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0" #환경마다 결과가 미세하게 달라지는 것을 방지

import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report

import _ml
from _style import setup

setup()

SEED = 42
print(tf.__version__, " 준비 완료")


[폰트] Malgun Gothic / unicode_minus=False
2.21.0  준비 완료


---
## 무엇을 예측할것인가?

### 주가 방향은 쓰지 않을것이다.

머신러닝에서 "오늘 오를까?"를 예측했다. 결과는 AUC 0.5부근으로 사실상 랜덤이였다.  
신호가 없는 문제로는 **알고리즘이 잘 동작하는지조차 확인할 수 없었다.**  

그래서 이번에는 신호가 있는 문제로 변경해서 보자.

```
오늘 거래량이 최근 20일 평균보다 많을까? (거래량 급증 여부)
```

거래량은 한 번 몰리면 며칠 이어지는 성질이 있어서 **어느정도는 예측이 된다.**  
급증을 미리 감지하면 뉴스/공시를 먼저 확인할 수 있다.


In [6]:
def load_dataset():
    """
    스케일이 큰 피처 두개를 더해서 돌려주는 함수

    volume_prev, close_prev를 만들어준다.
    거래량은 수십만~수천만, 종가는 수천~수십만 단위.
    등락률은(0.0x)과 섞이면 표준편차가 억단위로 벌어진다.
    => 스케일 차이가 큰 피처
    """

    base = _ml.build_dataset().sort_values(["code", "date"])
    g = base.groupby("code")
    base["volume_prev"] = g["volume"].shift(1) #어제 거래량
    base["close_prev"] = g["close"].shift(1) #어제 종가

    # 정답생성 : 오늘거래량 > 최근 20일 평균 거래량
    raw = _ml.load_merged().sort_values(["code", "date"])
    raw["vol_ma20_prev"] = (raw.groupby("code")["volume"]
                            .transform(lambda s: s.rolling(20).mean().shift(1)))
    raw["surge"] = (raw["volume"] > raw["vol_ma20_prev"]).astype("int8")

    df = base.merge(raw[["code", "date", "surge"]], on=["code", "date"], how="left")
    return df.dropna(subset=["volume_prev", "close_prev", "surge"]).reset_index(drop=True)

FEATURES = _ml.FEATURES + ["volume_prev", "close_prev"]

df = load_dataset()
print(f"데이터 {len(df):,}행, 피처{len(FEATURES)}개")
print(f"급증 비율 : {df['surge'].mean():.3f}")
print("\n피처 : ", FEATURES)

데이터 87,360행, 피처9개
급증 비율 : 0.366

피처 :  ['ret_1d', 'ret_5d', 'ma5_ratio', 'ma20_ratio', 'vol20', 'volume_ratio', 'range_pct', 'volume_prev', 'close_prev']


In [8]:
# 시계열 데이터는 시간순으로 분할
train, test, cutoff = _ml.time_split(df, test_ratio=0.2)

X_train = train[FEATURES].to_numpy("float32")
y_train = train["surge"].to_numpy("float32")
X_test = test[FEATURES].to_numpy("float32")
y_test = test["surge"].to_numpy("float32")

print(f"분할 기준일 : {cutoff.date()}")
print(f"학습 : {X_train.shape}  검증 : {X_test.shape}")

scale = pd.DataFrame({
    "평균" : X_train.mean(axis=0),
    "표준편차" : X_train.std(axis=0),
    "최대" : X_train.max(axis=0),
}, index=FEATURES)

print()

print(scale.to_string(float_format=lambda v: f"{v:>18,.4f}"))
print(f"\n표준편차 최대/최소 = {scale['표준편차'].max() / scale['표준편차'].min():,.0f} 배 차이")

분할 기준일 : 2026-01-16
학습 : (69960, 9)  검증 : (17400, 9)

                             평균               표준편차                 최대
ret_1d                  -0.0003             0.0263             0.2755
ret_5d                  -0.0014             0.0613             0.4257
ma5_ratio                0.9989             0.0298             1.2395
ma20_ratio               0.9950             0.0687             1.4317
vol20                    0.0243             0.0101             0.0677
volume_ratio             1.0016             0.9528            15.5041
range_pct                0.0435             0.0204             0.3370
volume_prev        577,769.7500     1,555,611.6250    71,626,144.0000
close_prev          46,903.8867        59,168.6992       552,685.0000

표준편차 최대/최소 = 154,746,496 배 차이


표준편차가 1억배 이상 차이나는 상태로 신경망에 입력하면, 거대 피처가 기울기 계산을 독점한다  
-> 학습이 제대로 되지 않기떄문에 표준화 또는 정규화의 전처리가 필수적이다.

---
## 딥러닝 3단계 : 정의 -> 컴파일 -> 학습

1. 정의 : 어떤 층을 어떻게 쌓을 것인가?  
2. 컴파일 : 무엇을 최소화하고 어떻게 할습할 것인가?  
3. 학습 : 데이터를 넣고 반복  

Keras(API)를 사용하면 전부 통일된 형태로 간단하게 구현이 가능하다.

In [10]:
def build_model(n_features, hidden=(64, 32), dropout=0.0):
    """
    이진 분류용 신경망 만들기.
    hidden : 은닉층 뉴런수
    dropout : 0보다 크면 각 은닉층 뒤에 Dropout을 넣는다.
    """

    # 가중치 초기값을 고정해서 동일한 데이터로 실습할 수 있게 하자.
    tf.keras.utils.set_random_seed(SEED)

    layers = [Input(shape=(n_features,))] #피처가 몇개인지
    for units in hidden:
        layers.append(Dense(units, activation="relu"))
        if dropout > 0:
            layers.append(Dropout(dropout))
    layers.append(Dense(1, activation="sigmoid")) #이진분류 -> 출력 1개 + Sigmoid

    model = Sequential(layers)

    model.compile(
        optimizer="adam", #학습률을 알아서 조절해준다.
        loss="binary_crossentropy", #손실함수 -> 오차를 측정하는 기준 : Sigmoid와 짝
        metrics=["accuracy"], #사람이 보려고 함께 계산
    )

    return model

model = build_model(len(FEATURES))
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 64)             │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,753 (10.75 KB)

 Trainable params: 2,753 (10.75 KB)

 Non-trainable params: 0 (0.00 B)

 **Total params: 2,753 ?**  

 Dense 층의 파라미터는 (입력개수 * 뉴런개수) + 뉴련개수(편향). 
 dense_3    9 x 64 + 64 = 640
 dense_4    64 x 32 + 32 = 2,080
 dense_5    32 x 1 + 1 = 33
 = 2,753  
   
Dense(64) 한줄로 가중치 576개와 편향 64개를 알아서 만든것이다.  
학습에 실제로 쓰이는 행은 검증용 20%를 제외한 값이다.

### 학습 곡선을 그리는 함수

In [12]:
def plot_history(hist, title, ax=None, ylim=None):
    """
    학습 곡선을 그리기

    train(loss)과 validation(val_loss) 두개를 겹처 그리는 차트.
    -> 과적합 여부를 알 수 있다.
    """

    if ax is None:
        _, ax = plt.Subplot(figsize=(7,3.6))

    # loss: 훈련 데이터 오차
    # val_loss: 검증 데이터의 오차
    ax.plot(hist.history["loss"], label="train", lw=1.6)
    ax.plot(hist.history["val_loss"], label="validation", lw=1.6)

    #검증 손실이 가장 낮았던 지점 => EarlyStopping지점
    best = int(np.argmin(hist.history["val_loss"]))
    ax.axvline(best, color="gray", ls=":", lw=1)
    ax.text(best, ax.get_ylim()[1], f" val_loss 최저 {best + 1}에포크",
            fontsize=8, color="gray", va="top")

    ax.set(xlabel="에포크", ylabel="손실", title=title)
    if ylim:
        ax.set_ylim(*ylim)    
    ax.legend()
    return ax

def evaluate(name, model, X, y, scaler=None):
    """테스트 셋 성능을 한줄로 출력."""
    Xe = scaler.transform(X) if scaler is not None else X
    prob = model.predict(Xe, verbose=0).ravel()
    auc = roc_auc_score(y, prob)
    acc = accuracy_score(y, (prob >= 0.5).astype(int))
    print(f"{name:<28} AUC {auc:.4f}  정확도 {acc:.4f}")
    return auc
    